# EquiNexus grouped validation baseline

Attach the Solar Filament Segmentation Challenge data in Kaggle and enable internet for the public GitHub checkout and pinned dependency check. This notebook records the source commit, detects the attached input, and runs the shared local-contrast baseline. The predictor and scorer run on CPU; a GPU accelerator is not needed for this baseline.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/BrownAssassin/EquiNexus.git"
REPOSITORY_REF = "main"
REPOSITORY_PATH = Path("/kaggle/working/EquiNexus")

if not (REPOSITORY_PATH / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(REPOSITORY_PATH)], check=True)
else:
    subprocess.run(["git", "-C", str(REPOSITORY_PATH), "fetch", "--depth", "1", "origin", REPOSITORY_REF], check=True)
    subprocess.run(["git", "-C", str(REPOSITORY_PATH), "checkout", "--detach", "FETCH_HEAD"], check=True)

SOURCE_COMMIT = subprocess.run(["git", "-C", str(REPOSITORY_PATH), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
sys.path.insert(0, str(REPOSITORY_PATH / "src"))
print(f"EquiNexus source commit: {SOURCE_COMMIT}")

In [ ]:
import importlib.metadata

requirements = REPOSITORY_PATH / "requirements.txt"
gpu_requirements = REPOSITORY_PATH / "requirements-gpu.txt"
mismatches = []
for line in [*requirements.read_text(encoding="utf-8").splitlines(), *gpu_requirements.read_text(encoding="utf-8").splitlines()]:
    line = line.strip()
    if not line or line.startswith("#") or line.startswith("--"):
        continue
    package, expected = line.split("==", 1)
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        mismatches.append((package, installed, expected))

if mismatches:
    print("Installing pinned packages:", mismatches)
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-cache-dir", "-r", str(requirements), "-r", str(gpu_requirements)], check=True)
    raise RuntimeError("Pinned packages were installed. Restart the Kaggle session, then run all cells again.")

print("Pinned package versions are present.")

In [ ]:
from filament_baseline.runner import run_validation

DATA_SEARCH_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/filament-validation-baseline")
RESULT = run_validation(
    repo_root=REPOSITORY_PATH,
    data_root=DATA_SEARCH_ROOT,
    output_root=OUTPUT_ROOT,
    overlay_count=12,
)
print("Validation PQ:", RESULT["metrics"]["global"]["pq"])
print("Split SHA-256:", RESULT["split"]["sha256"])
print("Run output:", OUTPUT_ROOT)

In [ ]:
from IPython.display import display, Image

for image_path in sorted((OUTPUT_ROOT / "overlays").glob("*.png"))[:6]:
    print(image_path.name)
    display(Image(filename=str(image_path)))

print((OUTPUT_ROOT / "metrics.json").read_text(encoding="utf-8"))